# Persiapan Dependencies dan GPU

In [1]:
import os, re

if "KAGGLE_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install rouge_score -q
!pip install torchao>=0.16.0 -q
!pip install langdetect -q

print("Semua library berhasil diinstall!")

Semua library berhasil diinstall!


In [2]:
import subprocess

libraries = [
    "torch", "torchao", "unsloth", "transformers",
    "trl", "peft", "datasets", "accelerate",
    "bitsandbytes", "huggingface_hub", "rouge_score",
    "langdetect", "sentencepiece",
]

print("VERSI LIBRARY")
for lib in libraries:
    try:
        result = subprocess.run(["pip", "show", lib], capture_output=True, text=True)
        for line in result.stdout.split("\n"):
            if line.startswith("Version:"):
                version = line.split(": ")[1].strip()
                print(f"{lib:<20} == {version}")
    except Exception:
        print(f"{lib:<20} -- tidak ditemukan")

VERSI LIBRARY
torch                == 2.10.0+cu128
torchao              == 0.10.0
unsloth              == 2026.6.9
transformers         == 4.56.2
trl                  == 0.22.2
peft                 == 0.18.1
datasets             == 4.3.0
accelerate           == 1.13.0
bitsandbytes         == 0.49.2
huggingface_hub      == 0.36.2
rouge_score          == 0.1.2
langdetect           == 1.0.9
sentencepiece        == 0.2.1


In [3]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

from unsloth.chat_templates import get_chat_template
from trl import GRPOTrainer, GRPOConfig
from transformers import TextStreamer
from datasets import load_dataset
from kaggle_secrets import UserSecretsClient
from rouge_score import rouge_scorer
from langdetect import detect, LangDetectException
import torch
import re
import numpy as np
import wandb
import os

if torch.cuda.is_available():
    gpu_stats        = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory       = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU Detected : {gpu_stats.name}")
    print(f"Max Memory   : {max_memory} GB")
    print(f"Used Memory  : {start_gpu_memory} GB")
else:
    print("WARNING: GPU Not Detected!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-06-23 19:39:37.561299: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782243577.584839    1316 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782243577.592700    1316 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782243577.612597    1316 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782243577.612619    1316 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782243577.612621    1316 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: UnslothAlignPropTrainer is already patched.
Unsloth: UnslothBCOTrainer is already patched.
Unsloth: UnslothCPOTrainer is already patched.
Unsloth: UnslothDDPOTrainer is already patched.
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGKDTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothIterativeSFTTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothNashMDTrainer is already patched.
Unsloth: UnslothOnlineDPOTrainer is already patched.
Unsloth: UnslothORPOTrainer is already patched.
Unsloth: UnslothPPOTrainer is already patched.
Unsloth: UnslothPRMTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Unsloth: UnslothRLOOTrainer is already patched.
Unsloth: UnslothSFTTrainer is already patched.
Unsloth: UnslothXPOTrainer is already patched.
GPU Detected : Tesla T4
Max Memory   : 14.562 GB
Used Memo

# Login Huggingface dan Setup WandB

In [4]:
from huggingface_hub import login

secrets = UserSecretsClient()

# Login Hugging Face
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Login HF berhasil!")

Login HF berhasil!


In [5]:
wandb_token = secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_API_KEY"] = wandb_token
print("WandB siap!")

WandB siap!


# Load Model Hasil Fine-tuning

In [6]:
MODEL_NAME     = "Rahmat15/qwen2.5-3b-indonesian-legal"
MAX_SEQ_LENGTH = 1024
DTYPE          = None
LOAD_IN_4BIT   = True

print(f"Memuat model hasil fine-tuning: {MODEL_NAME}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    load_in_4bit    = LOAD_IN_4BIT,
    load_in_8bit    = False,
    full_finetuning = False,
    dtype           = DTYPE,
)

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

# Memasang LoRA untuk GRPO training
model = FastLanguageModel.get_peft_model(
    model,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    r                          = 16,
    lora_alpha                 = 16,
    lora_dropout               = 0,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
)

print("\nModel berhasil dimuat!")
model.print_trainable_parameters()

Memuat model hasil fine-tuning: Rahmat15/qwen2.5-3b-indonesian-legal
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth 2026.6.9 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.



Model berhasil dimuat!
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


# Load Dataset

In [7]:
dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")

# Ambil subset untuk GRPO agar efisien
dataset_grpo = dataset.select(range(5000))
print(f"Dataset GRPO: {len(dataset_grpo)} baris")

SYSTEM_PROMPT = """Anda adalah asisten AI hukum Indonesia.
WAJIB gunakan format berikut di setiap jawaban:
<think>
[tuliskan analisis dan proses berpikir Anda di sini]
</think>
[tuliskan jawaban final dalam bahasa Indonesia]"""

def format_dataset_grpo(examples):
    inputs  = examples["input"]
    outputs = examples["output"]
    prompts = []
    answers = []

    few_shot_user = "Apakah karyawan kontrak berhak mendapat THR?"
    few_shot_asst = """<think>
Karyawan kontrak (PKWT) diatur dalam UU Ketenagakerjaan No.13 Tahun 2003.
Berdasarkan Permenaker No.6 Tahun 2016, karyawan PKWT yang telah bekerja
minimal 1 bulan berhak mendapat THR secara proporsional sesuai masa kerja.
</think>
Ya, karyawan kontrak berhak mendapat THR secara proporsional berdasarkan
Permenaker No.6 Tahun 2016."""

    for input_text, output in zip(inputs, outputs):
        messages = [
            {"role": "system",    "content": SYSTEM_PROMPT},
            # Few-shot: tunjukkan contoh format yang benar
            {"role": "user",      "content": few_shot_user},
            {"role": "assistant", "content": few_shot_asst},
            # Pertanyaan sebenarnya
            {"role": "user",      "content": input_text},
        ]
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize              = False,
            add_generation_prompt = True
        )
        prompts.append(prompt)
        answers.append(output)

    return {"prompt": prompts, "answer": answers}

dataset_grpo = dataset_grpo.map(format_dataset_grpo, batched=True)

print(f"\nDataset GRPO siap!")
print(f"\nContoh prompt:\n{dataset_grpo[1]['prompt']}")
print(f"\nContoh answer:\n{dataset_grpo[1]['answer']}")

Dataset GRPO: 5000 baris


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]


Dataset GRPO siap!

Contoh prompt:
<|im_start|>system
Anda adalah asisten AI hukum Indonesia.
WAJIB gunakan format berikut di setiap jawaban:
<think>
[tuliskan analisis dan proses berpikir Anda di sini]
</think>
[tuliskan jawaban final dalam bahasa Indonesia]<|im_end|>
<|im_start|>user
Apakah karyawan kontrak berhak mendapat THR?<|im_end|>
<|im_start|>assistant
<think>
Karyawan kontrak (PKWT) diatur dalam UU Ketenagakerjaan No.13 Tahun 2003.
Berdasarkan Permenaker No.6 Tahun 2016, karyawan PKWT yang telah bekerja
minimal 1 bulan berhak mendapat THR secara proporsional sesuai masa kerja.
</think>
Ya, karyawan kontrak berhak mendapat THR secara proporsional berdasarkan
Permenaker No.6 Tahun 2016.<|im_end|>
<|im_start|>user
Ciptakan cara unik dan lucu untuk menggambarkan presiden negaramu.
<|im_end|>
<|im_start|>assistant


Contoh answer:
Sebagai AI, saya tidak tergabung dalam satu negara dan karena itu tidak memiliki presiden tertentu yang harus digambarkan. Namun, saya dapat membantu A

# Reward Function

## Reward Function: Format \<think> yang benar

In [8]:
def format_reward_func(completions, **kwargs):
    """
    Reward shaping sesuai kriteria:
    +0.2 → ada tag <think>
    +0.3 → ada tag </think>
    +1.0 → format sempurna
    -0.5 → tag muncul lebih dari sekali
    """

    # Debug: print completion pertama setiap dipanggil
    if completions:
        print(f"\n[DEBUG] Sample completion:\n{completions[0][:200]}\n")
        
    rewards = []
    for completion in completions:
        score       = 0.0
        text        = completion.strip()
        open_count  = len(re.findall(r'<think>', text))
        close_count = len(re.findall(r'</think>', text))

        if open_count > 1 or close_count > 1:
            score -= 0.5
        else:
            if open_count == 1:
                score += 0.2   # sesuai kriteria

            if close_count == 1:
                score += 0.3   # sesuai kriteria

            # Cek format sempurna dengan isi think tidak kosong
            think_match    = re.search(r'<think>(.*?)</think>', text, re.DOTALL)
            think_is_empty = True
            if think_match:
                think_content  = think_match.group(1).strip()
                think_is_empty = len(think_content) == 0

            if not think_is_empty:
                perfect_pattern = r'^<think>\s*(\S.{10,}?)\s*</think>\s*\S+'
                if re.match(perfect_pattern, text, re.DOTALL):
                    score = 1.0   # override ke 1.0 jika sempurna

        rewards.append(score)
    return rewards

## Reward Function: Panjang reasoning \<think>

In [9]:
def reasoning_length_reward(completions, **kwargs):
    """
    Poin proporsional sesuai kriteria:
    +0.0 → tidak ada tag atau kosong
    +0.2 → < 50 karakter
    +0.5 → 50-199 karakter
    +1.0 → >= 200 karakter
    Toleran jika terpotong batas token
    """
    rewards = []
    for completion in completions:
        match = re.search(r'<think>(.*?)</think>', completion, re.DOTALL)
        if match:
            think_content = match.group(1)
        else:
            # Toleran jika terpotong
            match_open    = re.search(r'<think>(.*)', completion, re.DOTALL)
            think_content = match_open.group(1) if match_open else ""

        think_length = len(think_content.strip())

        if think_length == 0:
            score = 0.0
        elif think_length < 50:      # sesuai kriteria
            score = 0.2
        elif think_length < 200:     # sesuai kriteria
            score = 0.5
        else:                        # sesuai kriteria
            score = 1.0

        rewards.append(score)
    return rewards

## Reward Function: Kebenaran jawaban (ROUGE)

In [10]:
def correctness_reward(completions, answer, **kwargs):
    """
    +1.0 jika output mengandung ground truth atau ROUGE tinggi
    Sesuai kriteria: flat +1.0, bukan bertahap
    """
    scorer  = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
    rewards = []

    for completion, ground_truth in zip(completions, answer):
        if '</think>' in completion:
            final_answer = completion.split('</think>')[-1].strip()
        else:
            final_answer = completion.strip()

        # Cek 1: ground truth terkandung langsung
        if ground_truth.lower() in final_answer.lower():
            rewards.append(1.0)
            continue

        # Cek 2: ROUGE-L flat reward sesuai kriteria
        scores      = scorer.score(ground_truth, final_answer)
        rouge_score = scores['rougeL'].fmeasure

        if rouge_score >= 0.3:
            rewards.append(1.0)   # flat +1.0 sesuai kriteria
        else:
            rewards.append(0.0)   # flat +0.0

    return rewards

## Reward Function: Bahasa Indonesia

In [11]:
def language_reward_func(completions, **kwargs):
    """
    +1.0 → bahasa Indonesia
    -0.5 → bahasa Inggris
    Sesuai kriteria
    """
    rewards = []
    for completion in completions:
        if '</think>' in completion:
            final_answer = completion.split('</think>')[-1].strip()
        else:
            final_answer = completion.strip()

        try:
            if len(final_answer) < 10:
                rewards.append(0.0)
                continue
            lang = detect(final_answer)
            if lang == 'id':
                rewards.append(1.0)
            elif lang == 'en':
                rewards.append(-0.5)
            else:
                rewards.append(0.0)
        except LangDetectException:
            rewards.append(0.0)
    return rewards

## Test Reward Function

In [12]:
test_completions = [
    # Case 1: Format sempurna, reasoning panjang
    "<think>Ini adalah proses berpikir saya yang cukup panjang untuk menjawab pertanyaan ini dengan baik dan benar berdasarkan pengetahuan yang ada tentang hukum ketenagakerjaan Indonesia.</think> Jawaban finalnya adalah ini dalam bahasa Indonesia.",

    # Case 2: Tidak ada tag think
    "Jawaban langsung tanpa proses berpikir.",

    # Case 3: Tag berganda
    "<think>Pikiran 1</think> jawaban <think>Pikiran 2</think> jawaban lagi.",

    # Case 4: Think kosong
    "<think>   </think> Jawaban dengan think kosong.",

    # Case 5: Think pendek < 30 karakter
    "<think>Pikiran singkat.</think> Jawaban final.",

    # Case 6: Think sedang 30-100 karakter
    "<think>Pertanyaan ini menyangkut hak lembur karyawan.</think> Jawaban sedang.",
]

test_answers = [
    "Jawaban yang benar dalam bahasa Indonesia tentang hak karyawan.",
    "Jawaban referensi kedua tentang ketenagakerjaan.",
    "Jawaban referensi ketiga.",
    "Jawaban referensi keempat.",
    "Jawaban referensi kelima.",
    "Jawaban referensi keenam tentang hak lembur.",
]

print("TEST REWARD FUNCTIONS")

for i, (completion, answer) in enumerate(zip(test_completions, test_answers)):
    print(f"\n--- Test Case {i+1} ---")
    print(f"Completion : {completion[:70]}...")
    print(f"format_reward      : {format_reward_func([completion])[0]:>5.1f}")
    print(f"reasoning_length   : {reasoning_length_reward([completion])[0]:>5.1f}")
    print(f"correctness_reward : {correctness_reward([completion], [answer])[0]:>5.2f}")
    print(f"language_reward    : {language_reward_func([completion])[0]:>5.1f}")

print("\nTest selesai!")

TEST REWARD FUNCTIONS

--- Test Case 1 ---
Completion : <think>Ini adalah proses berpikir saya yang cukup panjang untuk menjaw...

[DEBUG] Sample completion:
<think>Ini adalah proses berpikir saya yang cukup panjang untuk menjawab pertanyaan ini dengan baik dan benar berdasarkan pengetahuan yang ada tentang hukum ketenagakerjaan Indonesia.</think> Jawaban 

format_reward      :   1.0
reasoning_length   :   0.5
correctness_reward :  1.00
language_reward    :   1.0

--- Test Case 2 ---
Completion : Jawaban langsung tanpa proses berpikir....

[DEBUG] Sample completion:
Jawaban langsung tanpa proses berpikir.

format_reward      :   0.0
reasoning_length   :   0.0
correctness_reward :  0.00
language_reward    :   1.0

--- Test Case 3 ---
Completion : <think>Pikiran 1</think> jawaban <think>Pikiran 2</think> jawaban lagi...

[DEBUG] Sample completion:
<think>Pikiran 1</think> jawaban <think>Pikiran 2</think> jawaban lagi.

format_reward      :  -0.5
reasoning_length   :   0.2
correctness_rew

# Training GRPO

In [14]:
wandb.init(
    project = "PGABL-Qwen2.5-3B",
    name    = "GRPO_qwen2.5_legal_v3",
    reinit  = "finish_previous",
)

print("Memulai GRPO Training v3...")

grpo_config = GRPOConfig(
    max_steps                   = 150,
    num_generations             = 6,
    max_completion_length       = 512,   
    max_prompt_length           = 768,  
    learning_rate               = 1e-5,  
    warmup_steps                = 30,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    save_strategy               = "steps",
    save_steps                  = 100,
    save_total_limit            = 2,
    output_dir                  = "/kaggle/working/grpo_results_v4",
    logging_steps               = 10,
    report_to                   = "wandb",
    run_name                    = "GRPO_qwen2.5_legal_v4",
    seed                        = 42,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
)

trainer_grpo = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    args             = grpo_config,
    train_dataset    = dataset_grpo,
    reward_funcs     = [
        format_reward_func,
        reasoning_length_reward,
        correctness_reward,
        language_reward_func,
    ],
)

trainer_grpo.train()
wandb.finish()
print("\nGRPO Training v3 selesai!")

profiling/Time taken: UnslothGRPOTrainer._calculate_rewards,█▃▁
profiling/Time taken: UnslothGRPOTrainer._prepare_inputs,█▁▁▁▆▁▁▁▄▁▁▁
profiling/Time taken: UnslothGRPOTrainer.correctness_reward,█▃▁
profiling/Time taken: UnslothGRPOTrainer.format_reward_func,▄▁█
profiling/Time taken: UnslothGRPOTrainer.language_reward_func,▁▄█
profiling/Time taken: UnslothGRPOTrainer.reasoning_length_reward,▅█▁
profiling/Time taken: UnslothGRPOTrainer.transformers.generate,▆█▁
train/global_step,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
profiling/Time taken: UnslothGRPOTrainer._calculate_rewards,0.12322
profiling/Time taken: UnslothGRPOTrainer._prepare_inputs,0.0
profiling/Time taken: UnslothGRPOTrainer.correctness_reward,0.05626


Memulai GRPO Training v3...
Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 6


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 150
O^O/ \_/ \    Batch size per device = 6 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (6 x 4 x 1) = 24
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)



[DEBUG] Sample completion:
Wabah Hitam mengubah Eropa secara dramatis dengan menyebabkan kematian jutaan orang dan menimbulkan perubahan signifikan dalam gaya hidup, budaya, dan ekonomi Eropa. Berikut adalah beberapa dampak pen



Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_func / mean,rewards / format_reward_func / std,rewards / reasoning_length_reward / mean,rewards / reasoning_length_reward / std,rewards / correctness_reward / mean,rewards / correctness_reward / std,rewards / language_reward_func / mean,rewards / language_reward_func / std
10,0.000000,1.289583,0.187421,177.937506,13.600000,453.700000,0.033333,167.616185,13.600000,422.100000,0.000021,0.000000,0.000000,0.000000,0.000000,0.312500,0.460366,0.977083,0.093381
20,0.000000,1.229167,0.163379,188.550006,32.800000,437.400000,0.037500,176.848100,32.800000,413.000000,0.000398,0.000000,0.000000,0.000000,0.000000,0.245833,0.309166,0.983333,0.069058
30,0.000000,1.304167,0.206908,162.970837,18.700000,455.000000,0.070833,137.669561,18.700000,412.500000,0.000531,0.002083,0.010206,0.004167,0.020412,0.337500,0.421119,0.960417,0.084555
40,0.000000,1.220833,0.171665,198.141673,28.600000,451.500000,0.079167,174.642914,28.600000,434.400000,0.003203,0.004167,0.020412,0.002083,0.010206,0.254167,0.399672,0.960417,0.090461
50,0.000000,1.133333,0.187456,153.458340,18.300000,412.600000,0.041667,139.507120,18.300000,376.700000,0.007668,0.000000,0.000000,0.000000,0.000000,0.250000,0.388834,0.883333,0.180387
60,0.000000,1.277083,0.234670,156.679171,14.000000,412.800000,0.025000,148.403231,14.000000,401.000000,0.009366,0.000000,0.000000,0.000000,0.000000,0.312500,0.340830,0.964583,0.133460
70,0.000000,1.272917,0.080760,207.358344,15.600000,466.900000,0.070833,186.772539,15.600000,432.700000,0.008709,0.000000,0.000000,0.000000,0.000000,0.291667,0.424620,0.981250,0.050675
80,0.000000,1.218750,0.265521,198.500002,24.100000,457.900000,0.041667,185.172797,24.100000,444.000000,0.019260,0.000000,0.000000,0.000000,0.000000,0.275000,0.433721,0.943750,0.202907
90,0.000000,1.277083,0.226024,181.987508,22.000000,436.600000,0.041667,169.417862,22.000000,409.000000,0.023863,0.000000,0.000000,0.000000,0.000000,0.316667,0.416752,0.960417,0.132926
100,0.000000,1.141667,0.090544,174.729174,21.400000,452.500000,0.025000,167.187472,21.400000,429.700000,0.029244,0.000000,0.000000,0.000000,0.000000,0.179167,0.298442,0.962500,0.092878



[DEBUG] Sample completion:
Kekulé menemukan bahwa senyawa bernama benzena memiliki struktur cincin benzena dengan enam atom karbon yang terhubung satu sama lain dalam susunan heksagonal yang teratur dengan ikatan tunggal dan ga


[DEBUG] Sample completion:
Steve Jobs dikenal sebagai pencipta iPod.


[DEBUG] Sample completion:
John Doe
(123) 456-7890
jdoe@email.com

Kepemimpinan dan Komunikasi: 
- Memiliki keahlian komunikasi yang kuat untuk memfasilitasi kerjasama tim dan mengambil keputusan dengan efektif.
- Mampu menyamp


[DEBUG] Sample completion:
```python
# Membuat Array dengan 5 elemen
my_array = [4, 7, 2, 9, 1]

# Mengurutkan Array secara menaik
my_array.sort()

# Menampilkan Array yang sudah diurutkan secara menaik
print(my_array)
```

Out


[DEBUG] Sample completion:
1. Kecepatan: Robot memiliki kemampuan untuk melakukan tugas-tugas tertentu dengan kecepatan yang signifikan lebih cepat dibandingkan manusia. Ini membuat mereka ideal untuk tugas-tugas yang membutuhk


[DEBUG] 

profiling/Time taken: UnslothGRPOTrainer._calculate_rewards,▄█▂▅▅▂▃▂▄▃▂▅▃▂▃▅▆▅▆▅▃▅▃▄▁▆▄▂▂▅▂▇▃▅▂▂▄▅▄▁
profiling/Time taken: UnslothGRPOTrainer._prepare_inputs,▁█▁▁▁▇▁▁█▁▆▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▆▁▅▁▁▁
profiling/Time taken: UnslothGRPOTrainer.correctness_reward,▅▃▃▄▅▄▃▆▅█▄▆▄▂▃▂▄▃▁▂▂▅▃▃▂▅▂▆▃▃▄▄▃▄▂▄▂▂▅▅
profiling/Time taken: UnslothGRPOTrainer.format_reward_func,▂▃█▂▃█▄▆▆▂▂█▇▃▂▃▄▂▄▆█▅▄▇▄▅▅▃▃▆▁▃▂▄▃▄▂▄▂▃
profiling/Time taken: UnslothGRPOTrainer.language_reward_func,▄▅▆▄▅▄▄▅▃▂▆▃▅▂▁▄▄▅▆▃▄▃▆▅▅▃█▇▅▁▄▄▄▄▄▄▄▄▃▇
profiling/Time taken: UnslothGRPOTrainer.reasoning_length_reward,▄▇▆▅▇▄▆▅▅▆▆▅▂▅█▆▄▆▄▃▆█▃▆▄▃▁▅▄▃▅▅▆▅▇▇▃▅▂▃
profiling/Time taken: UnslothGRPOTrainer.transformers.generate,███▆▇███▆▇██▃████▇▇▁▆▇▇██▇██▆▆██▆▅█▅▄██▃
train/clip_ratio/high_max,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/high_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/low_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+28,...



GRPO Training v3 selesai!


# Upload Model GRPO ke HuggingFace

In [15]:
GRPO_REPO_NAME = "Rahmat15/qwen2.5-3b-indonesian-legal-grpo"

print(f"Mengupload model GRPO ke: {GRPO_REPO_NAME}")

model.push_to_hub_merged(
    GRPO_REPO_NAME,
    tokenizer   = tokenizer,
    save_method = "merged_16bit",
    token       = True
)

print(f"\nUpload selesai!")
print(f"https://huggingface.co/{GRPO_REPO_NAME}")
print("\nTambahkan link ini ke file link_huggingface.txt!")

Mengupload model GRPO ke: Rahmat15/qwen2.5-3b-indonesian-legal-grpo


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 2 files from cache to `Rahmat15/qwen2.5-3b-indonesian-legal-grpo`: 100%|██████████| 2/2 [00:05<00:00,  2.92s/it]


...

Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [01:24<01:24, 84.82s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:07<00:00, 63.94s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/Rahmat15/qwen2.5-3b-indonesian-legal-grpo`

Upload selesai!
https://huggingface.co/Rahmat15/qwen2.5-3b-indonesian-legal-grpo

Tambahkan link ini ke file link_huggingface.txt!
